# Rung-1 comparison: le5, le6, le6_hot

Tasks: fastapi_15589,fastapi_13920,fastapi_14258,fastapi_11194,fastapi_14487,fastapi_15280,fastapi_14419,fastapi_15800,fastapi_14306,rich_3882,rich_3470,rich_4079,rich_3521,rich_3934,rich_3468,rich_3180

Generated by scripts/make_kaggle_notebook.py. Attach competition data, the wheelhouse dataset and the 31B model; accelerator L4 ×4; Run All; stop the session when done.

In [ ]:
# 1. Install the competition wheelhouse BEFORE anything imports google.adk
#    (Kaggle preinstalls another google-adk; importing it first causes pydantic class clashes).
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not WHEELHOUSE_DIR.exists():
    WHEELHOUSE_DIR = Path(glob.glob('/kaggle/input/**/gemma-4-developer-agent-wheelhouse', recursive=True)[0])

for pth in glob.glob('/usr/local/lib/python*/*-packages/*cutlass*.pth'):
    try:
        os.unlink(pth)
    except OSError:
        pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
assert wheels, f'No wheels found in {WHEELHOUSE_DIR}'
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

In [ ]:
# 2. Write the submissions to evaluate
import re
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
SUB_ROOT = WORKING_DIR / 'submissions'
TASK_IDS = ['fastapi_15589', 'fastapi_13920', 'fastapi_14258', 'fastapi_11194', 'fastapi_14487', 'fastapi_15280', 'fastapi_14419', 'fastapi_15800', 'fastapi_14306', 'rich_3882', 'rich_3470', 'rich_4079', 'rich_3521', 'rich_3934', 'rich_3468', 'rich_3180']
CONFIGS = ['le5', 'le6', 'le6_hot']
REPEATS = 1
CONCURRENCY = 2  # tasks evaluated at the same time (each has its own workspace and /tmp)
EMBEDDED = {
 "le5": {
  "agent.yaml": "# Arm le5 (rung-3k), from le3 (rung-3i): locator -> editor rebuilt on flat tools. The scout skill is gone (its nested args were garbled\n# in 25-50% of calls, rung-3f); the locator reads with run_command/read_file, writes /tmp/plan.md with a run_command\n# heredoc, calls the editor, checks once, submits. Prompt in the bf/mix step-by-step style with r4's decision rule.\nname: locator\ndescription: Finds what to change, plans it for the editor, checks the result and submits.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/locator.md\ntools:\n  - run_command\n  - read_file\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: sub_agents/editor.yaml\ngenerate_content_config: !include configs/locator_sampling.yaml\n",
  "configs/editor_sampling.yaml": "# Editor (rung-3i le3): low temperature for exact copying, small thinking budget (it applies a plan, no design).\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 512\n",
  "configs/locator_sampling.yaml": "# Locator (le5): mix's sampling with thinking 1024 instead of 2048 (thinking used most of the 8 minutes in le3).\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 1024\n",
  "eval_config.yaml": "# Arm le5 (rung-3k): le3_c28 (28 calls, warning at 18) + no scratch scripts + locator thinking 1024.\nevaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 90\n",
  "prompts/editor.md": "You are the editor. A locator has saved a plan in /tmp/plan.md. Apply it exactly; do not explore, redesign or test.\n\n1. Read the plan: run_command with cat /tmp/plan.md\n2. For each FILE and CHANGE: call edit_file with old_string = the exact old lines from the plan and new_string = the\n   new lines. If edit_file returns an error, read the real lines once (read_file with start_line and end_line around\n   them) and retry with the text exactly as it is in the file, or edit with run_command:\n   python3 - << 'EOF'\n   p = 'path/to/file.py'\n   s = open(p).read()\n   old = '''exact old lines'''\n   new = '''new lines'''\n   assert s.count(old) == 1\n   open(p, 'w').write(s.replace(old, new))\n   EOF\n3. Reply with one line per file and nothing else:\n   EDITED: path/to/file.py: what changed\n   FAILED: path/to/file.py: why (quote the error)\n\nNever edit tests, conftest.py or configuration files. Call tools only through the tool-calling interface.\n",
  "prompts/locator.md": "You are the locator in a two-agent team resolving one issue in a repository, fast and minimally. You find what must\nchange, write a plan, and check the result; the editor makes the edit. Nobody will answer questions.\n\n## How you are scored, and the one decision rule\nHidden tests run on the final diff, even if you run out of time; no edit scores zero. As soon as you can say which\nlines to change and how, your very next tool call writes the plan, and the call after that is the editor. Never\nre-read, search again or write a script first. Checking comes after the edit.\n\n## Step 0: Extract the contract (before any tool call)\nFrom the problem statement, list the exact function names, class names, error messages, exception types, HTTP status\ncodes, CLI flags, and parameter names it mentions, copied character-for-character. Hidden tests exercise exactly\nthese names and spellings. Ignore template boilerplate in the issue: HTML comments, checklists, AI disclaimers.\n\n## Step 1: Locate (calls 1-8)\nThe first user message already contains the workspace layout, so never spend a call on ls or find to orient.\n- Search the most specific identifier or error string: git grep -n \"text\" -- '*.py' | head -20\n- read_file with start and end line, at most 120 lines per read. Never read whole files.\n- Feature request: find the closest existing sibling feature and read it end to end (signature, imports, __all__\n  exports, dispatch or option tables, registration).\n- For documentation-code tasks (e.g. FastAPI), the code to change is under docs_src/.\n- Fix the behaviour where it is produced, not where it is observed.\n\n## Step 2: Write the plan (the decision rule above)\nWrite it with one run_command, exactly in this shape (the quoted PLAN marker keeps every character as written):\ncat > /tmp/plan.md << 'PLAN'\nFILE: path/to/file.py\nWHAT IT DOES: one line about the function you change\nCHANGE: replace these exact lines, copied from read_file with their indentation:\n<old lines>\nWITH:\n<new lines>\nCONTRACT: the exact names, messages or values the tests expect\nPLAN\n- Repeat FILE ... CONTRACT for each file. Keep the old lines short (3-10 lines) and unique; never docstrings.\n- When you add a check, an error or an option, copy how the neighbouring code does it (assert or raise, the same\n  exception class, the same message style).\n\n## Step 3: Call the editor\nCall the editor tool with the request \"Apply /tmp/plan.md\". It answers EDITED or FAILED for each file.\n\n## Step 4: One check, at most one correction, then submit\n- run_command: git diff --stat && git diff | head -60\n- Run ONLY the targeted test for the file you changed:\n  python3 -m pytest tests/test_target.py -x -q -k name -p no:anyio -o timeout=0 > /tmp/t1.log 2>&1; tail -n 25 /tmp/t1.log\n- Correct once (rewrite /tmp/plan.md, call the editor again) only if the diff is not what you planned, the editor\n  answered FAILED, or a test fails because of an error in the new code. A failing test that asserts the old behaviour\n  the task asks to change is expected: keep the change.\n- Then call submit_patch. A local green test is not the grader: submit even if the check is red.\n\n## Rules\n- When a tool result says only a few tool calls remain: if no edit is in yet, write the plan and call the editor now;\n  if it is in, call submit_patch.\n- Don't write scripts to reproduce the issue or to experiment: read the code. Files you leave in the repository become\n  part of the patch and fail grading.\n- submit_patch and get_status are free. If get_status shows under 90 seconds remaining, call submit_patch immediately.\n- Never run bare pytest or full-repo suites. Never modify, create or delete tests or test configuration (tests/,\n  conftest.py, pytest.ini, pyproject.toml, setup.cfg, tox.ini).\n- Never search outside /workspace. Never refactor unrelated code. Call tools only through the tool-calling interface.\n\n## Example (an invented package)\nTask: \"Client should reject a negative timeout.\" git grep finds pkg/client.py line 88, self.timeout = timeout, and two\nlines below it raise ValueError(\"retries must be >= 0\") for retries. Found it: the next call writes the plan (add the\nsame kind of check raising ValueError(\"timeout must be >= 0\")), then the editor, then the client tests, then submit:\n8 calls.\n",
  "sub_agents/editor.yaml": "name: editor\ndescription: >-\n  Applies the plan saved in /tmp/plan.md to the source files. Request: one line, \"Apply /tmp/plan.md\".\n  Answers one line per file: EDITED (path and what changed) or FAILED (path and why).\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/editor.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\ngenerate_content_config: !include ../configs/editor_sampling.yaml\n"
 },
 "le6": {
  "agent.yaml": "# Arm le6 (rung-3l): le5 + a counted reader. A setup step installs .git/agent/read.py (outside the patch); the\n# locator reads only through it via run_command (flat string). After 16 reads it refuses while calls are left, so the\n# hand-off fits the budget. The plan travels inside the editor request (no /tmp/plan.md, 2 counted calls saved).\nagent_class: SequentialAgent\nname: setup_then_locate\ndescription: Installs the reader, then the locator finds the change and hands the plan to the editor.\nsub_agents:\n  - config_path: sub_agents/setup.yaml\n  - config_path: sub_agents/locator.yaml\n",
  "configs/editor_sampling.yaml": "# Editor (rung-3i le3): low temperature for exact copying, small thinking budget (it applies a plan, no design).\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 512\n",
  "configs/locator_sampling.yaml": "# Locator (le6): as le5 (T 0.2, thinking 1024).\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 1024\n",
  "configs/setup_sampling.yaml": "# Setup step (le6): one tool call, no reasoning needed.\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 512\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 128\n",
  "eval_config.yaml": "# Arm le6 (rung-3l): 28 calls (warning at 18; reader refuses after 16 reads), 8 min, 90 turns, as le5.\nevaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 90\n",
  "prompts/editor.md": "You are the editor. The request you receive is the plan from the locator. Apply it exactly; do not explore, redesign\nor test.\n\n1. For each FILE and CHANGE in the request: call edit_file with old_string = the old lines and new_string = the new\n   lines, exactly as written in the plan (no line numbers).\n2. If edit_file returns an error, read the real lines once (read_file with start_line and end_line around them) and\n   retry with the text exactly as it is in the file, or edit with run_command:\n   python3 - << 'EOF'\n   p = 'path/to/file.py'\n   s = open(p).read()\n   old = '''exact old lines'''\n   new = '''new lines'''\n   assert s.count(old) == 1\n   open(p, 'w').write(s.replace(old, new))\n   EOF\n3. Reply with one line per file and nothing else:\n   EDITED: path/to/file.py: what changed\n   FAILED: path/to/file.py: why (quote the error)\n\nNever edit tests, conftest.py or configuration files. Call tools only through the tool-calling interface.\n",
  "prompts/locator.md": "You are the locator in a two-agent team resolving one issue in a repository, fast and minimally. You find what must\nchange and hand the plan to the editor, then check the result; the editor makes the edit. Nobody will answer questions.\n\n## How you are scored, and the one decision rule\nHidden tests run on the final diff, even if you run out of time; no edit scores zero. As soon as you can say which\nlines to change and how, your very next tool call is the editor with the plan. Never re-read, search again or write\na script first. Checking comes after the edit.\n\n## Step 0: Extract the contract (before any tool call)\nFrom the problem statement, list the exact function names, class names, error messages, exception types, HTTP status\ncodes, CLI flags, and parameter names it mentions, copied character-for-character. Hidden tests exercise exactly\nthese names and spellings. Ignore template boilerplate in the issue: HTML comments, checklists, AI disclaimers.\n\n## Step 1: Locate with the reader\nThe reader is already installed. Read the code only through it, with run_command:\n  python3 .git/agent/read.py find \"most specific name or error message\"   (the hits and the code after them)\n  python3 .git/agent/read.py show path/to/file.py 40 160                   (numbered lines, at most 150)\n  python3 .git/agent/read.py history NAME                                   (commits that added or removed NAME;\n                                                                             useful when the task is only a title)\n- You have 16 reads; every result shows the count. At the limit the reader refuses: then call the editor.\n- If the reader is missing (\"No such file\"), read with git grep -n and sed -n '40,160p' path instead.\n- The first user message already contains the workspace layout, so never spend a call on ls or find to orient.\n- Feature request: find the closest existing sibling feature and read it end to end (signature, imports, __all__\n  exports, dispatch or option tables, registration).\n- For documentation-code tasks (e.g. FastAPI), the code to change is under docs_src/.\n- Fix the behaviour where it is produced, not where it is observed.\n\n## Step 2: Call the editor with the plan (the decision rule above)\nThe editor's request is the plan itself, exactly in this shape:\nFILE: path/to/file.py\nWHAT IT DOES: one line about the function you change\nCHANGE: replace these exact lines, copied from the reader with their indentation (without the line numbers):\n<old lines>\nWITH:\n<new lines>\nCONTRACT: the exact names, messages or values the tests expect\n- Repeat FILE ... CONTRACT for each file. Keep the old lines short (3-10 lines) and unique; never docstrings.\n- When you add a check, an error or an option, copy how the neighbouring code does it (assert or raise, the same\n  exception class, the same message style).\nThe editor answers EDITED or FAILED for each file.\n\n## Step 3: One check, at most one correction, then submit\n- run_command: git diff --stat && git diff | head -60\n- Run ONLY the targeted test for the file you changed:\n  python3 -m pytest tests/test_target.py -x -q -k name -p no:anyio -o timeout=0 > /tmp/t1.log 2>&1; tail -n 25 /tmp/t1.log\n- Correct once (call the editor again with the corrected plan) only if the diff is not what you planned, the editor\n  answered FAILED, or a test fails because of an error in the new code. A failing test that asserts the old behaviour\n  the task asks to change is expected: keep the change.\n- Then call submit_patch. A local green test is not the grader: submit even if the check is red.\n\n## Rules\n- When a tool result says only a few tool calls remain: if no edit is in yet, call the editor with your plan now;\n  if it is in, call submit_patch.\n- Don't write scripts to reproduce the issue or to experiment: read the code. Files you leave in the repository become\n  part of the patch and fail grading.\n- submit_patch and get_status are free. If get_status shows under 90 seconds remaining, call submit_patch immediately.\n- Never run bare pytest or full-repo suites. Never modify, create or delete tests or test configuration (tests/,\n  conftest.py, pytest.ini, pyproject.toml, setup.cfg, tox.ini).\n- Never search outside /workspace. Never refactor unrelated code. Call tools only through the tool-calling interface.\n\n## Example (an invented package)\nTask: \"Client should reject a negative timeout.\" read.py find \"self.timeout\" shows pkg/client.py line 88,\nself.timeout = timeout, and two lines below it raise ValueError(\"retries must be >= 0\") for retries. Found it: the\nnext call is the editor with the plan (add the same kind of check raising ValueError(\"timeout must be >= 0\")), then\nthe client tests, then submit: 6 calls.\n",
  "prompts/setup.md": "Your only job is to install the reading tool for the next agent. Do not read code or work on the task.\n\nCall run_skill_script with skill_name reader and file_path scripts/install.py, with no other arguments. Then reply\nwith one line, INSTALLED, and stop. If the call fails, call it once more exactly the same way, then reply and stop.\n",
  "skills/reader/SKILL.md": "---\nname: reader\ndescription: Installs the locator's reading tool at .git/agent/read.py (outside the patch). find, show and history share a limit of 16 reads; at the limit it refuses and says to call the editor.\n---\n\nInstall once per task: call run_skill_script with skill_name reader and file_path scripts/install.py (no other\narguments). It prints INSTALLED and how to use the reader through run_command.\n",
  "skills/reader/scripts/install.py": "\"\"\"Copy read.py into <repository>/.git/agent/ (git never puts .git in the patch) and reset its read counter.\"\"\"\nimport inspect\nimport os\nimport shutil\nimport sys\n\n\ndef workspace():\n    \"\"\"The repository root. The skill wrapper chdirs into a temp dir, so look for the directory it started in.\"\"\"\n    for frame in inspect.stack():\n        cwd = frame.frame.f_locals.get(\"_orig_cwd\")\n        if isinstance(cwd, str) and os.path.isdir(os.path.join(cwd, \".git\")):\n            return cwd\n    for cand in (os.environ.get(\"PYTHONPATH\", \"\").split(os.pathsep)[0], \"/workspace\", os.getcwd()):\n        if cand and os.path.isdir(os.path.join(cand, \".git\")):\n            return cand\n    sys.exit(\"reader: repository not found\")\n\n\nhere = os.path.dirname(os.path.abspath(__file__))\ntarget_dir = os.path.join(workspace(), \".git\", \"agent\")\nos.makedirs(target_dir, exist_ok=True)\nshutil.copyfile(os.path.join(here, \"read.py\"), os.path.join(target_dir, \"read.py\"))\ncounter = os.path.join(target_dir, \"reads\")\nif os.path.exists(counter):\n    os.remove(counter)\nprint(\"INSTALLED .git/agent/read.py. Read the code with run_command:\\n\"\n      '  python3 .git/agent/read.py find \"exact name or message\"\\n'\n      \"  python3 .git/agent/read.py show path/to/file.py 40 160\\n\"\n      \"  python3 .git/agent/read.py history NAME\\n\"\n      \"16 reads in total; every result shows the count.\")\n",
  "skills/reader/scripts/read.py": "\"\"\"The locator's reader, run through run_command (one flat string, so the tool call is never garbled).\n\n  python3 .git/agent/read.py find \"exact name or message\"     git grep hits with the code that follows\n  python3 .git/agent/read.py show path/to/file.py 40 160       numbered lines, at most 150 (also file.py:40-160)\n  python3 .git/agent/read.py history NAME | SHA                commits that added/removed NAME, or one commit\n\nAll three share LIMIT reads. At the limit it refuses and tells the locator to call the editor, while the task still\nhas tool calls left for the edit. Always exits 0 so the result reaches the model as normal output.\n\"\"\"\nimport os\nimport re\nimport subprocess\nimport sys\n\nLIMIT = 16\nWARN_FROM = 13\nHERE = os.path.dirname(os.path.abspath(__file__))\nWS = os.path.dirname(os.path.dirname(HERE))        # <repo>/.git/agent/read.py -> <repo>\nCOUNTER = os.path.join(HERE, \"reads\")\nUSAGE = ('usage: python3 .git/agent/read.py find \"NAME\" | show path/to/file.py START END | history NAME')\n\n\ndef run(cmd, timeout=60):\n    try:\n        res = subprocess.run(cmd, cwd=WS, shell=True, capture_output=True, text=True, timeout=timeout)\n        return (res.stdout + res.stderr).rstrip()\n    except subprocess.TimeoutExpired:\n        return f\"(timed out after {timeout}s)\"\n\n\ndef take_read():\n    used = int(open(COUNTER).read() or 0) if os.path.exists(COUNTER) else 0\n    if used >= LIMIT:\n        print(f\"READING LIMIT REACHED ({LIMIT} reads used). Stop reading. Call the editor now: the request is your plan \"\n              \"(FILE / WHAT IT DOES / CHANGE ... WITH ... / CONTRACT). After it answers, check the diff and submit.\")\n        sys.exit(0)\n    used += 1\n    with open(COUNTER, \"w\") as f:\n        f.write(str(used))\n    return used\n\n\ndef footer(used):\n    left = LIMIT - used\n    msg = f\"[read {used} of {LIMIT}\"\n    if used >= WARN_FROM:\n        msg += f\"; {left} left: decide the change and call the editor\" if left else \"; none left: call the editor now\"\n    return msg + \"]\"\n\n\ndef quote(text):\n    return \"'\" + text.replace(\"'\", \"'\\\\''\") + \"'\"\n\n\ndef main(argv):\n    if not argv:\n        print(USAGE)\n        return\n    mode, rest = argv[0].lower().lstrip(\"-\"), argv[1:]\n    if mode == \"find\":\n        query = \" \".join(rest).strip()\n        if not query:\n            print(USAGE)\n            return\n        used = take_read()\n        out = run(f\"git grep -n -I -A 12 -e {quote(query)} -- '*.py' '*.md' '*.txt' '*.toml' | head -70\")\n        if not out:\n            out = run(f\"git grep -n -I -i -e {quote(query)} -- . | head -30\") or \"(no match)\"\n        print(out)\n        print(footer(used))\n    elif mode == \"show\":\n        if not rest:\n            print(USAGE)\n            return\n        path, nums = rest[0], [a for a in rest[1:]]\n        m = re.match(r\"^(.*?):(\\d+)(?:-(\\d+))?$\", path)\n        if m:\n            path, nums = m.group(1), [m.group(2)] + ([m.group(3)] if m.group(3) else [])\n        nums = [int(x) for x in re.findall(r\"\\d+\", \" \".join(nums))]\n        path = path.removeprefix(\"/workspace/\").removeprefix(\"./\")\n        start = max(nums[0] if nums else 1, 1)\n        end = nums[1] if len(nums) > 1 and nums[1] >= start else start + 149\n        end = min(end, start + 149)\n        full = os.path.join(WS, path)\n        if not os.path.isfile(full):\n            print(f\"(no such file: {path})\")\n            return\n        used = take_read()\n        lines = open(full, encoding=\"utf-8\", errors=\"replace\").read().split(\"\\n\")\n        print(\"\\n\".join(f\"{i:5}| {lines[i - 1]}\" for i in range(start, min(end, len(lines)) + 1)))\n        print(footer(used))\n    elif mode == \"history\":\n        query = \" \".join(rest).strip()\n        if not query:\n            print(USAGE)\n            return\n        used = take_read()\n        if re.fullmatch(r\"[0-9a-f]{7,40}\", query):\n            print(run(f\"git show --stat --format='%h %s' {query} | head -20 && git show --format= {query} | head -80\"))\n        else:\n            print(run(f\"git log --oneline -S{quote(query)} | head -15\") or \"(no commit adds or removes it)\")\n            print(\"recent commits:\\n\" + run(\"git log --oneline -8\"))\n        print(footer(used))\n    else:\n        print(USAGE)\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1:])\n",
  "sub_agents/editor.yaml": "name: editor\ndescription: >-\n  Applies a plan to the source files. Request: the full plan (FILE / WHAT IT DOES / CHANGE ... WITH ... / CONTRACT\n  for each file). Answers one line per file: EDITED (path and what changed) or FAILED (path and why).\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/editor.md\ntools:\n  - read_file\n  - edit_file\n  - run_command\ngenerate_content_config: !include ../configs/editor_sampling.yaml\n",
  "sub_agents/locator.yaml": "name: locator\ndescription: Finds what to change, hands the plan to the editor, checks the result and submits.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/locator.md\ntools:\n  - run_command\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: editor.yaml\ngenerate_content_config: !include ../configs/locator_sampling.yaml\n",
  "sub_agents/setup.yaml": "name: setup\ndescription: Installs the locator's reader. Makes exactly one call.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/setup.md\ntools:\n  - get_status\nskills:\n  - skills/reader\ngenerate_content_config: !include ../configs/setup_sampling.yaml\n"
 },
 "le6_hot": {
  "agent.yaml": "# Arm le6 (rung-3l): le5 + a counted reader. A setup step installs .git/agent/read.py (outside the patch); the\n# locator reads only through it via run_command (flat string). After 16 reads it refuses while calls are left, so the\n# hand-off fits the budget. The plan travels inside the editor request (no /tmp/plan.md, 2 counted calls saved).\nagent_class: SequentialAgent\nname: setup_then_locate\ndescription: Installs the reader, then the locator finds the change and hands the plan to the editor.\nsub_agents:\n  - config_path: sub_agents/setup.yaml\n  - config_path: sub_agents/locator.yaml\n",
  "configs/editor_sampling.yaml": "# Editor (le6_hot): T 1.0, top_k 64, thinking 256 (le's original editor) instead of le6's T 0.2 / thinking 512.\ntemperature: 1.0\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 1024\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 256\n",
  "configs/locator_sampling.yaml": "# Locator (le6_hot): r4_budget-style fast and decisive sampling (T 1.0, top_k 64, thinking 512) instead of le6's\n# careful T 0.2 / thinking 1024.\ntemperature: 1.0\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 512\n",
  "configs/setup_sampling.yaml": "# Setup step (le6): one tool call, no reasoning needed.\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 512\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 128\n",
  "eval_config.yaml": "# Arm le6_hot (rung-3l): le6 with fast sampling (locator T 1.0 / thinking 512, editor T 1.0 / thinking 256). Only change.\nevaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 90\n",
  "prompts/editor.md": "You are the editor. The request you receive is the plan from the locator. Apply it exactly; do not explore, redesign\nor test.\n\n1. For each FILE and CHANGE in the request: call edit_file with old_string = the old lines and new_string = the new\n   lines, exactly as written in the plan (no line numbers).\n2. If edit_file returns an error, read the real lines once (read_file with start_line and end_line around them) and\n   retry with the text exactly as it is in the file, or edit with run_command:\n   python3 - << 'EOF'\n   p = 'path/to/file.py'\n   s = open(p).read()\n   old = '''exact old lines'''\n   new = '''new lines'''\n   assert s.count(old) == 1\n   open(p, 'w').write(s.replace(old, new))\n   EOF\n3. Reply with one line per file and nothing else:\n   EDITED: path/to/file.py: what changed\n   FAILED: path/to/file.py: why (quote the error)\n\nNever edit tests, conftest.py or configuration files. Call tools only through the tool-calling interface.\n",
  "prompts/locator.md": "You are the locator in a two-agent team resolving one issue in a repository, fast and minimally. You find what must\nchange and hand the plan to the editor, then check the result; the editor makes the edit. Nobody will answer questions.\n\n## How you are scored, and the one decision rule\nHidden tests run on the final diff, even if you run out of time; no edit scores zero. As soon as you can say which\nlines to change and how, your very next tool call is the editor with the plan. Never re-read, search again or write\na script first. Checking comes after the edit.\n\n## Step 0: Extract the contract (before any tool call)\nFrom the problem statement, list the exact function names, class names, error messages, exception types, HTTP status\ncodes, CLI flags, and parameter names it mentions, copied character-for-character. Hidden tests exercise exactly\nthese names and spellings. Ignore template boilerplate in the issue: HTML comments, checklists, AI disclaimers.\n\n## Step 1: Locate with the reader\nThe reader is already installed. Read the code only through it, with run_command:\n  python3 .git/agent/read.py find \"most specific name or error message\"   (the hits and the code after them)\n  python3 .git/agent/read.py show path/to/file.py 40 160                   (numbered lines, at most 150)\n  python3 .git/agent/read.py history NAME                                   (commits that added or removed NAME;\n                                                                             useful when the task is only a title)\n- You have 16 reads; every result shows the count. At the limit the reader refuses: then call the editor.\n- If the reader is missing (\"No such file\"), read with git grep -n and sed -n '40,160p' path instead.\n- The first user message already contains the workspace layout, so never spend a call on ls or find to orient.\n- Feature request: find the closest existing sibling feature and read it end to end (signature, imports, __all__\n  exports, dispatch or option tables, registration).\n- For documentation-code tasks (e.g. FastAPI), the code to change is under docs_src/.\n- Fix the behaviour where it is produced, not where it is observed.\n\n## Step 2: Call the editor with the plan (the decision rule above)\nThe editor's request is the plan itself, exactly in this shape:\nFILE: path/to/file.py\nWHAT IT DOES: one line about the function you change\nCHANGE: replace these exact lines, copied from the reader with their indentation (without the line numbers):\n<old lines>\nWITH:\n<new lines>\nCONTRACT: the exact names, messages or values the tests expect\n- Repeat FILE ... CONTRACT for each file. Keep the old lines short (3-10 lines) and unique; never docstrings.\n- When you add a check, an error or an option, copy how the neighbouring code does it (assert or raise, the same\n  exception class, the same message style).\nThe editor answers EDITED or FAILED for each file.\n\n## Step 3: One check, at most one correction, then submit\n- run_command: git diff --stat && git diff | head -60\n- Run ONLY the targeted test for the file you changed:\n  python3 -m pytest tests/test_target.py -x -q -k name -p no:anyio -o timeout=0 > /tmp/t1.log 2>&1; tail -n 25 /tmp/t1.log\n- Correct once (call the editor again with the corrected plan) only if the diff is not what you planned, the editor\n  answered FAILED, or a test fails because of an error in the new code. A failing test that asserts the old behaviour\n  the task asks to change is expected: keep the change.\n- Then call submit_patch. A local green test is not the grader: submit even if the check is red.\n\n## Rules\n- When a tool result says only a few tool calls remain: if no edit is in yet, call the editor with your plan now;\n  if it is in, call submit_patch.\n- Don't write scripts to reproduce the issue or to experiment: read the code. Files you leave in the repository become\n  part of the patch and fail grading.\n- submit_patch and get_status are free. If get_status shows under 90 seconds remaining, call submit_patch immediately.\n- Never run bare pytest or full-repo suites. Never modify, create or delete tests or test configuration (tests/,\n  conftest.py, pytest.ini, pyproject.toml, setup.cfg, tox.ini).\n- Never search outside /workspace. Never refactor unrelated code. Call tools only through the tool-calling interface.\n\n## Example (an invented package)\nTask: \"Client should reject a negative timeout.\" read.py find \"self.timeout\" shows pkg/client.py line 88,\nself.timeout = timeout, and two lines below it raise ValueError(\"retries must be >= 0\") for retries. Found it: the\nnext call is the editor with the plan (add the same kind of check raising ValueError(\"timeout must be >= 0\")), then\nthe client tests, then submit: 6 calls.\n",
  "prompts/setup.md": "Your only job is to install the reading tool for the next agent. Do not read code or work on the task.\n\nCall run_skill_script with skill_name reader and file_path scripts/install.py, with no other arguments. Then reply\nwith one line, INSTALLED, and stop. If the call fails, call it once more exactly the same way, then reply and stop.\n",
  "skills/reader/SKILL.md": "---\nname: reader\ndescription: Installs the locator's reading tool at .git/agent/read.py (outside the patch). find, show and history share a limit of 16 reads; at the limit it refuses and says to call the editor.\n---\n\nInstall once per task: call run_skill_script with skill_name reader and file_path scripts/install.py (no other\narguments). It prints INSTALLED and how to use the reader through run_command.\n",
  "skills/reader/scripts/install.py": "\"\"\"Copy read.py into <repository>/.git/agent/ (git never puts .git in the patch) and reset its read counter.\"\"\"\nimport inspect\nimport os\nimport shutil\nimport sys\n\n\ndef workspace():\n    \"\"\"The repository root. The skill wrapper chdirs into a temp dir, so look for the directory it started in.\"\"\"\n    for frame in inspect.stack():\n        cwd = frame.frame.f_locals.get(\"_orig_cwd\")\n        if isinstance(cwd, str) and os.path.isdir(os.path.join(cwd, \".git\")):\n            return cwd\n    for cand in (os.environ.get(\"PYTHONPATH\", \"\").split(os.pathsep)[0], \"/workspace\", os.getcwd()):\n        if cand and os.path.isdir(os.path.join(cand, \".git\")):\n            return cand\n    sys.exit(\"reader: repository not found\")\n\n\nhere = os.path.dirname(os.path.abspath(__file__))\ntarget_dir = os.path.join(workspace(), \".git\", \"agent\")\nos.makedirs(target_dir, exist_ok=True)\nshutil.copyfile(os.path.join(here, \"read.py\"), os.path.join(target_dir, \"read.py\"))\ncounter = os.path.join(target_dir, \"reads\")\nif os.path.exists(counter):\n    os.remove(counter)\nprint(\"INSTALLED .git/agent/read.py. Read the code with run_command:\\n\"\n      '  python3 .git/agent/read.py find \"exact name or message\"\\n'\n      \"  python3 .git/agent/read.py show path/to/file.py 40 160\\n\"\n      \"  python3 .git/agent/read.py history NAME\\n\"\n      \"16 reads in total; every result shows the count.\")\n",
  "skills/reader/scripts/read.py": "\"\"\"The locator's reader, run through run_command (one flat string, so the tool call is never garbled).\n\n  python3 .git/agent/read.py find \"exact name or message\"     git grep hits with the code that follows\n  python3 .git/agent/read.py show path/to/file.py 40 160       numbered lines, at most 150 (also file.py:40-160)\n  python3 .git/agent/read.py history NAME | SHA                commits that added/removed NAME, or one commit\n\nAll three share LIMIT reads. At the limit it refuses and tells the locator to call the editor, while the task still\nhas tool calls left for the edit. Always exits 0 so the result reaches the model as normal output.\n\"\"\"\nimport os\nimport re\nimport subprocess\nimport sys\n\nLIMIT = 16\nWARN_FROM = 13\nHERE = os.path.dirname(os.path.abspath(__file__))\nWS = os.path.dirname(os.path.dirname(HERE))        # <repo>/.git/agent/read.py -> <repo>\nCOUNTER = os.path.join(HERE, \"reads\")\nUSAGE = ('usage: python3 .git/agent/read.py find \"NAME\" | show path/to/file.py START END | history NAME')\n\n\ndef run(cmd, timeout=60):\n    try:\n        res = subprocess.run(cmd, cwd=WS, shell=True, capture_output=True, text=True, timeout=timeout)\n        return (res.stdout + res.stderr).rstrip()\n    except subprocess.TimeoutExpired:\n        return f\"(timed out after {timeout}s)\"\n\n\ndef take_read():\n    used = int(open(COUNTER).read() or 0) if os.path.exists(COUNTER) else 0\n    if used >= LIMIT:\n        print(f\"READING LIMIT REACHED ({LIMIT} reads used). Stop reading. Call the editor now: the request is your plan \"\n              \"(FILE / WHAT IT DOES / CHANGE ... WITH ... / CONTRACT). After it answers, check the diff and submit.\")\n        sys.exit(0)\n    used += 1\n    with open(COUNTER, \"w\") as f:\n        f.write(str(used))\n    return used\n\n\ndef footer(used):\n    left = LIMIT - used\n    msg = f\"[read {used} of {LIMIT}\"\n    if used >= WARN_FROM:\n        msg += f\"; {left} left: decide the change and call the editor\" if left else \"; none left: call the editor now\"\n    return msg + \"]\"\n\n\ndef quote(text):\n    return \"'\" + text.replace(\"'\", \"'\\\\''\") + \"'\"\n\n\ndef main(argv):\n    if not argv:\n        print(USAGE)\n        return\n    mode, rest = argv[0].lower().lstrip(\"-\"), argv[1:]\n    if mode == \"find\":\n        query = \" \".join(rest).strip()\n        if not query:\n            print(USAGE)\n            return\n        used = take_read()\n        out = run(f\"git grep -n -I -A 12 -e {quote(query)} -- '*.py' '*.md' '*.txt' '*.toml' | head -70\")\n        if not out:\n            out = run(f\"git grep -n -I -i -e {quote(query)} -- . | head -30\") or \"(no match)\"\n        print(out)\n        print(footer(used))\n    elif mode == \"show\":\n        if not rest:\n            print(USAGE)\n            return\n        path, nums = rest[0], [a for a in rest[1:]]\n        m = re.match(r\"^(.*?):(\\d+)(?:-(\\d+))?$\", path)\n        if m:\n            path, nums = m.group(1), [m.group(2)] + ([m.group(3)] if m.group(3) else [])\n        nums = [int(x) for x in re.findall(r\"\\d+\", \" \".join(nums))]\n        path = path.removeprefix(\"/workspace/\").removeprefix(\"./\")\n        start = max(nums[0] if nums else 1, 1)\n        end = nums[1] if len(nums) > 1 and nums[1] >= start else start + 149\n        end = min(end, start + 149)\n        full = os.path.join(WS, path)\n        if not os.path.isfile(full):\n            print(f\"(no such file: {path})\")\n            return\n        used = take_read()\n        lines = open(full, encoding=\"utf-8\", errors=\"replace\").read().split(\"\\n\")\n        print(\"\\n\".join(f\"{i:5}| {lines[i - 1]}\" for i in range(start, min(end, len(lines)) + 1)))\n        print(footer(used))\n    elif mode == \"history\":\n        query = \" \".join(rest).strip()\n        if not query:\n            print(USAGE)\n            return\n        used = take_read()\n        if re.fullmatch(r\"[0-9a-f]{7,40}\", query):\n            print(run(f\"git show --stat --format='%h %s' {query} | head -20 && git show --format= {query} | head -80\"))\n        else:\n            print(run(f\"git log --oneline -S{quote(query)} | head -15\") or \"(no commit adds or removes it)\")\n            print(\"recent commits:\\n\" + run(\"git log --oneline -8\"))\n        print(footer(used))\n    else:\n        print(USAGE)\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1:])\n",
  "sub_agents/editor.yaml": "name: editor\ndescription: >-\n  Applies a plan to the source files. Request: the full plan (FILE / WHAT IT DOES / CHANGE ... WITH ... / CONTRACT\n  for each file). Answers one line per file: EDITED (path and what changed) or FAILED (path and why).\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/editor.md\ntools:\n  - read_file\n  - edit_file\n  - run_command\ngenerate_content_config: !include ../configs/editor_sampling.yaml\n",
  "sub_agents/locator.yaml": "name: locator\ndescription: Finds what to change, hands the plan to the editor, checks the result and submits.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/locator.md\ntools:\n  - run_command\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: editor.yaml\ngenerate_content_config: !include ../configs/locator_sampling.yaml\n",
  "sub_agents/setup.yaml": "name: setup\ndescription: Installs the locator's reader. Makes exactly one call.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/setup.md\ntools:\n  - get_status\nskills:\n  - skills/reader\ngenerate_content_config: !include ../configs/setup_sampling.yaml\n"
 }
}

shutil.rmtree(SUB_ROOT, ignore_errors=True)
for name, files in EMBEDDED.items():
    for rel, text in files.items():
        path = SUB_ROOT / name / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf-8')

if 'sample' in CONFIGS:
    # Google's sample as shipped, minus its LoRA adapters.
    sample = SUB_ROOT / 'sample'
    shutil.copytree(DATA_DIR / 'sample_submission', sample)
    shutil.rmtree(sample / 'adapters', ignore_errors=True)
    for yml in list(sample.rglob('*.yaml')):
        yml.write_text(re.sub(r'^adapter:.*\n', '', yml.read_text(), flags=re.M))
    (sample / 'eval_config.yaml').write_text(
        'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 100\n  max_time_minutes: 5.0\n  max_turns: 90\n')

for name in CONFIGS:
    print(name, sorted(p.relative_to(SUB_ROOT / name).as_posix() for p in (SUB_ROOT / name).rglob('*') if p.is_file()))

In [ ]:
# 3. Start vLLM once for all configs (no adapters → LoRA off → more KV cache). Takes ~6–8 min.
import litellm, torch
from adk_submission import VllmConfig, VllmServer

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
# The model must be attached as notebook input (Add Input -> Models -> google/gemma-4, variant TARGET_MODEL_NAME).
_cands = sorted(p for p in glob.glob(f'/kaggle/input/models/google/gemma-4/**/{TARGET_MODEL_NAME}/*', recursive=True)
                if Path(p, 'config.json').exists())
assert _cands, f'Model {TARGET_MODEL_NAME} not found under /kaggle/input: attach it as a notebook input'
MODEL_PATH = Path(_cands[-1])  # latest version
print('MODEL_PATH', MODEL_PATH)
# Per-model vLLM flags (host, forum 748287; the scorer has the same): the 12B checkpoint is labelled
# gemma4_unified, which vLLM 0.19.1 cannot load, so vLLM is told to load it as the regular gemma4 text model.
MODEL_VLLM_EXTRA_ARGS: dict[str, tuple[str, ...]] = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
# LOCAL TEST WORKAROUND (the scorer runs its own vLLM): with that flag, vLLM 0.19.1 (wheelhouse v30) still crashes
# on the 12B (KeyError 'vision_embedder.patch_dense.weight'). gemma4.py already skips the multimodal weights by name
# (audio_tower., vision_tower., embed_audio., embed_vision.), but the 12B file names 9 of its image weights
# model.vision_embedder.*, which is not in that list. Add it, nothing else: every other tensor name in the 12B file
# (checked from its safetensors header: model.language_model.*, 48 standard gemma4 layers) loads as before, and any
# other unknown name would still fail loudly.
if TARGET_MODEL_NAME == 'gemma-4-12b-it-qat-w4a16-ct':
    import importlib.util  # locate the file without importing vLLM's model code in this process
    _src_path = Path(importlib.util.find_spec('vllm').origin).parent / 'model_executor' / 'models' / 'gemma4.py'
    _src = _src_path.read_text()
    _old = '            "embed_vision.",\n'
    if 'SKIP_VISION_EMBEDDER' not in _src:
        assert _src.count(_old) == 1, 'gemma4.py skip list not found: vLLM changed, do not run the 12B'
        _src_path.write_text(_src.replace(_old, _old + '            "vision_embedder.",  # SKIP_VISION_EMBEDDER\n'))
    print('gemma4.py skip list now includes vision_embedder.')
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=list(MODEL_VLLM_EXTRA_ARGS.get(TARGET_MODEL_NAME, ())),
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
models = server_instance.create_model_registry(aliases=[TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')

# Installed harness versions (the wheelhouse changes over time; rung-1i notes).
from importlib.metadata import version, PackageNotFoundError
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
print('VERSIONS', {p: _ver(p) for p in ('adk-submission', 'swegemma', 'adk-eval-core', 'google-adk', 'vllm')})

# Sampling diagnostic (rung-1j): replay the REAL first orchestrator request (full 1b prompt + harness task message)
# under several sampling settings, thinking off. Counts hidden reasoning, thought markers leaking into text, and
# whether a proper tool call came back. Task text is read at runtime from the competition data (never committed).
def _sampling_diagnostic(task_ids=("rich_3882", "fastapi_15589"), samples=3):
    import json as _json, time, types, urllib.request as _url
    from swegemma.harness.agent_runner import build_agent_prompt
    _base = server_instance.base_url.rstrip("/")
    _base = _base if _base.endswith("/v1") else _base + "/v1"
    _served = _json.load(_url.urlopen(_base + "/models"))["data"][0]["id"]
    def _fn(name, desc, props=None, req=()):
        return {"type": "function", "function": {"name": name, "description": desc, "parameters":
                {"type": "object", "properties": props or {}, "required": list(req)}}}
    _s = {"type": "string"}; _i = {"type": "integer"}
    _tools = [_fn("run_command", "Run a shell command in the workspace.", {"command": _s}, ["command"]),
              _fn("read_file", "Read a file.", {"filepath": _s, "start_line": _i, "end_line": _i}, ["filepath"]),
              _fn("get_status", "Budget and progress."), _fn("submit_patch", "Submit the current diff."),
              _fn("search_similar_code", "Semantic code search.", {"query": _s}, ["query"]),
              _fn("get_code_neighbors", "Graph neighbors of a symbol.", {"symbol": _s}, ["symbol"]),
              _fn("get_code_subgraph", "Graph around a symbol.", {"symbol": _s}, ["symbol"]),
              _fn("executor", "Makes the code change.", {"request": _s}, ["request"]),
              _fn("verifier", "Checks the change.", {"request": _s}, ["request"])]
    _tasks = {}
    for _line in (DATA_DIR / "tasks.jsonl").read_text().splitlines():
        _d = _json.loads(_line)
        if _d.get("instance_id") in task_ids:
            _tasks[_d["instance_id"]] = types.SimpleNamespace(**_d)
    class _NS(types.SimpleNamespace):
        def __getattr__(self, _name):  # any field the harness reads but we did not set -> None
            return None
    _cfg = _NS(budget=_NS(time_minutes=5.0, tool_calls=80, turns=90), harness=_NS(command_timeout_seconds=240, max_stdout_chars=5000),
               enable_sandbox_testing=True)
    _instr = (SUB_ROOT / "1b" / "prompts" / "orchestrator.md").read_text()
    _settings = [("T1.0 p0.95 k64", 1.0, 0.95, 64), ("T1.0 p0.80 k20", 1.0, 0.80, 20), ("T0.6 p0.95 k64", 0.6, 0.95, 64),
                 ("T0.4 p0.95 k64", 0.4, 0.95, 64), ("T0.2 p0.95 k64", 0.2, 0.95, 64)]
    _summary = {}
    for _tid, _task in _tasks.items():
        try:
            _user = build_agent_prompt(_task, _cfg, "", declared_tools=[t["function"]["name"] for t in _tools])
        except Exception as _e:
            print("build_agent_prompt failed, using plain task text:", repr(_e))
            _user = "Problem Statement:\n" + _task.problem_statement
        _msgs = [{"role": "system", "content": _instr.replace("{problem_description}", _task.problem_statement)},
                 {"role": "user", "content": _user}]
        for _name, _t, _p, _k in _settings:
            for _rep in range(samples):
                _body = {"model": _served, "messages": _msgs, "tools": _tools, "temperature": _t, "top_p": _p,
                         "top_k": _k, "max_tokens": 1024, "chat_template_kwargs": {"enable_thinking": False}}
                _t0 = time.time()
                _req = _url.Request(_base + "/chat/completions", data=_json.dumps(_body).encode(),
                                    headers={"Content-Type": "application/json"})
                _r = _json.load(_url.urlopen(_req, timeout=600))
                _m = _r["choices"][0]["message"]
                _reason = _m.get("reasoning_content") or _m.get("reasoning") or ""
                _content = _m.get("content") or ""
                _ok = bool(_m.get("tool_calls"))
                _leak = "<|channel>" in _content or "<channel|>" in _content
                _row = (time.time() - _t0, _r["usage"]["completion_tokens"], len(_reason), _leak, _ok)
                _summary.setdefault(_name, []).append(_row)
                print(f"{_tid:14} {_name}  #{_rep + 1}: {_row[0]:5.1f}s  out_tok={_row[1]:5}  reasoning_chars={_row[2]:5}  "
                      f"marker_in_text={_leak}  tool_call={_ok}  prompt_tok={_r['usage']['prompt_tokens']}")
                if (_reason or _leak) and _rep == 0:
                    print("    starts:", repr((_reason or _content)[:160]))
    print("SAMPLING SUMMARY (per setting over all tasks and samples)")
    for _name, _rows in _summary.items():
        _n = len(_rows)
        print(f"  {_name}: mean {sum(r[0] for r in _rows) / _n:5.1f}s  mean out_tok {sum(r[1] for r in _rows) / _n:6.0f}  "
              f"with reasoning {sum(r[2] > 0 for r in _rows)}/{_n}  marker leaks {sum(r[3] for r in _rows)}/{_n}  "
              f"proper tool calls {sum(r[4] for r in _rows)}/{_n}")
RUN_SAMPLING_DIAGNOSTIC = False  # rung-1k answered it: direct calls never think
try:
    if RUN_SAMPLING_DIAGNOSTIC:
        _sampling_diagnostic()
except Exception as _e:  # never block the evaluation
    print("sampling diagnostic failed:", repr(_e))

In [ ]:
# 4. Evaluate every config on the same tasks (Evaluator configured like the scorer; see scripts/run_eval.py)
import asyncio, concurrent.futures, time, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

DATA = DATA_DIR  # default used by build_config (copied from scripts/run_eval.py)


def read_eval_config(submission_dir):
    """The four eval_config.yaml keys the scorer reads (defaults as in the host notebook)."""
    path = Path(submission_dir) / "eval_config.yaml"
    raw = yaml.safe_load(path.read_text()) if path.exists() else {}
    section = (raw or {}).get("evaluation", raw or {})
    return {
        "timeout_seconds": int(section.get("timeout_seconds", 300)),
        "max_tool_calls": section.get("max_tool_calls"),
        "max_time_minutes": float(section.get("max_time_minutes", 5.0)),
        "max_turns": section.get("max_turns"),
    }


def wheels_dir_for(data_dir):
    """Official test wheels, plus data/wheels_extra if present (local only: test deps missing from the public set,
    forum 745775). Merged as links into data/wheels_merged so the official folder stays untouched."""
    official, extra = Path(data_dir) / "wheels", Path(data_dir) / "wheels_extra"
    if not extra.is_dir():
        return official
    merged = Path(data_dir) / "wheels_merged"
    merged.mkdir(exist_ok=True)
    for wheel in [*official.glob("*.whl"), *extra.glob("*.whl")]:
        link = merged / wheel.name
        if not link.exists():
            link.hardlink_to(wheel)
    return merged


def build_config(submission_dir, task_ids, results_dir, models, data_dir=DATA, sandbox="docker",
                 adapter_manifest=None, display_mode="single", concurrency=1):
    limits, gen_constraints = build_submission_limits()
    budgets = read_eval_config(submission_dir)
    return EvalConfig(
        tasks_path=Path(data_dir) / "tasks.jsonl",
        snapshots_dir=Path(data_dir) / "snapshots",
        results_dir=Path(results_dir),
        submission_dir=Path(submission_dir),
        models=models,
        sandbox=sandbox,
        wheels_dir=wheels_dir_for(data_dir),
        graph_dir=str(Path(data_dir) / "graphs"),
        embeddings_dir=str(Path(data_dir) / "embeddings"),
        task_ids=list(task_ids),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapter_manifest,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5,
        ),
        display_mode=display_mode,
        concurrency=concurrency,
        **budgets,
    )


def run_sync(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro)).result()
    return asyncio.run(coro)


RESULTS = WORKING_DIR / 'results'
shutil.rmtree(RESULTS, ignore_errors=True)
# Repeats are interleaved (r1 of every config, then r2, ...) so a session that dies early still has complete pairs.
RUNS = [(name, f'{name}_r{rep + 1}' if REPEATS > 1 else name) for rep in range(REPEATS) for name in CONFIGS]
wall = {}
for name, key in RUNS:
    start = time.time()
    config = build_config(SUB_ROOT / name, TASK_IDS, RESULTS / key, models, data_dir=DATA_DIR,
                          sandbox='subprocess', display_mode='single' if CONCURRENCY == 1 else 'quiet',
                          concurrency=CONCURRENCY)
    result = run_sync(Evaluator(config).run())
    wall[key] = time.time() - start
    print(f'==> {key}: {result.resolved}/{result.total} resolved in {wall[key] / 60:.1f} min')
    shutil.make_archive(str(WORKING_DIR / 'results'), 'zip', RESULTS)  # partial results survive a crash

In [ ]:
# 5. Comparison table + results.zip (download it from the Output panel), then STOP THE SESSION.
import json
from collections import Counter


NUDGE_PREFIXES = ("Please continue your work", "Your previous response reached the token limit")


HELPER_FORMATS = {"reader": "LOCATION:", "executor": "CHANGED:", "verifier": "VERDICT:"}


TEST_PATHS = ("tests/", "test_", "conftest.py", "pytest.ini")


def bucket(task, patch=""):
    """Coarse failure bucket from the harness result (patch text comes from patches/<id>.patch)."""
    error = (task.get("error") or "").lower()
    patch = patch or task.get("agent_patch") or ""
    if task.get("resolved"):
        return "resolved"
    if "context" in error and ("window" in error or "length" in error):
        return "overflow"
    if "not found" in error and "tool" in error:
        return "unknown tool"
    if not patch.strip():
        return "timeout, no patch" if "timeout" in error or "timed out" in error else "no patch"
    changed = [line.split(" b/")[-1] for line in patch.splitlines() if line.startswith("diff --git")]
    if any(any(t in f for t in TEST_PATHS) for f in changed):
        return "edited tests"
    return "wrong fix"


def trace_stats(trace):
    steps = trace.get("steps", [])
    stats = Counter()
    tools = Counter()
    commands = Counter()
    helper_ok = Counter()
    helper_total = Counter()
    last_prompt = {}  # per agent: helpers have their own conversations
    peak_prompt = 0
    for step in steps:
        message = str(step.get("message") or "")
        if step.get("source") != "agent" and message.startswith(NUDGE_PREFIXES):
            stats["nudges"] += 1
        metrics = step.get("metrics") or {}
        prompt = metrics.get("prompt_tokens")
        if prompt:
            stats["model_calls"] += 1
            stats["output_tokens"] += metrics.get("completion_tokens") or 0
            peak_prompt = max(peak_prompt, prompt)
            # Compaction replaces old events with a summary: the agent's next prompt is much smaller than its last.
            # A helper (agent_tool) starts a fresh conversation on every call, so a new call resets its baseline.
            author = (step.get("extra") or {}).get("author")
            previous = last_prompt.get(author)
            if previous and prompt < 0.7 * previous:
                stats["compactions"] += 1
            last_prompt[author] = prompt
        for call in step.get("tool_calls") or []:
            name = call.get("function_name")
            tools[name] += 1
            if name == "run_command":
                commands[(call.get("arguments") or {}).get("command", "")] += 1
            if name in HELPER_FORMATS:
                last_prompt.pop(name, None)
        # Observations are tagged with the tool that produced them. Helper (agent_tool) replies arrive as
        # {"raw": "<final text>"} in the step whose observation is tagged with the helper's name.
        observation = step.get("observation") or {}
        source = (observation.get("extra") or {}).get("tool_name")
        content = str(observation.get("content", ""))
        if '"status": "error"' in content[:40] or "'status': 'error'" in content[:40] or content.startswith('{"error"'):
            stats["tool_errors"] += 1
        if source in HELPER_FORMATS:
            helper_total[source] += 1
            try:
                reply = json.loads(content).get("raw", content)
            except (ValueError, AttributeError):
                reply = content
            if HELPER_FORMATS[source] in reply[:300]:
                helper_ok[source] += 1
            if "<|tool_call>" in reply:
                stats["broken_calls"] += 1
        if "<|tool_call>" in message:
            stats["broken_calls"] += 1
    stats["peak_prompt"] = peak_prompt
    stats["repeated_cmds"] = sum(n - 1 for n in commands.values() if n > 1)
    helpers = {h: f"{helper_ok[h]}/{helper_total[h]}" for h in helper_total}
    return stats, tools, helpers


def analyze(results_dir):
    results_dir = Path(results_dir)
    tasks = [json.loads(line) for line in (results_dir / "task_results.jsonl").read_text().splitlines() if line]
    print(f"\n## {results_dir.name}: {sum(t.get('resolved', False) for t in tasks)}/{len(tasks)} resolved")
    header = (f"{'task':<16} {'outcome':<18} {'sec':>5} {'calls':>5} {'out_tok':>7} {'peak_in':>7} "
              f"{'compact':>7} {'nudge':>5} {'repeat':>6} {'errors':>6} {'broken':>6}  tools / helpers")
    print(header)
    print("-" * len(header))
    for task in tasks:
        task_id = task.get("instance_id") or task.get("task_id")
        trace_file = results_dir / "traces" / f"trace_{task_id}.json"
        trace = json.loads(trace_file.read_text()) if trace_file.exists() else {}
        stats, tools, helpers = trace_stats(trace)
        patch_file = results_dir / "patches" / f"{task_id}.patch"
        patch = patch_file.read_text() if patch_file.exists() else ""
        tool_list = ", ".join(f"{name}×{n}" for name, n in tools.most_common())
        helper_list = (" | format ok " + ", ".join(f"{h} {v}" for h, v in helpers.items())) if helpers else ""
        print(f"{task_id:<16} {bucket(task, patch):<18} {task.get('duration_seconds', 0):>5.0f} {stats['model_calls']:>5} "
              f"{stats['output_tokens']:>7} {stats['peak_prompt']:>7} {stats['compactions']:>7} "
              f"{stats['nudges']:>5} {stats['repeated_cmds']:>6} {stats['tool_errors']:>6} {stats['broken_calls']:>6}  "
              f"{tool_list}{helper_list}")
        if patch:
            changed = [line.split(" b/")[-1] for line in patch.splitlines() if line.startswith("diff --git")]
            print(f"{'':<16} patch: {', '.join(changed)}")
        if task.get("error"):
            print(f"{'':<16} error: {task['error'][:150]}")


for _, key in RUNS:
    analyze(RESULTS / key)
shutil.make_archive(str(WORKING_DIR / 'results'), 'zip', RESULTS)
print('\nWall clock per config (min):', {k: round(v / 60, 1) for k, v in wall.items()})
print('Wrote /kaggle/working/results.zip')